In [ ]:
import sys
import os
from pathlib import Path

notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent
src_dir      = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from config_loader import load_config
import triage_filter

cfg = load_config()

# Resolve triage path (new key — created by run_triage if absent)
triage_path = Path(cfg["paths"]["triage"])

print("Config loaded.")
print("  raw_images :", cfg["paths"]["raw_images"])
print("  triage     :", triage_path)


In [ ]:
# Loads SigLIP ViT-SO400M-14-SigLIP-384 via open_clip.
# Downloads ~3 GB of weights on first run (cached by HuggingFace Hub).
model, processor, device = triage_filter.load_siglip_model(cfg)
print(f"SigLIP ready on: {device}")


In [ ]:
# Test mode: process only the first 10 patent folders.
# Remove limit=10 (or set limit=None) to run on the full dataset.
triage_filter.run_triage(cfg, threshold=0.60, limit=10)


In [ ]:
import json
import pandas as pd

triage_path = Path(cfg["paths"]["triage"])
summary_csv = triage_path / "triage_summary.csv"

if not summary_csv.exists():
    print("No triage_summary.csv found — run Cell 3 first.")
else:
    summary_df = pd.read_csv(summary_csv)

    total_images  = summary_df["total_images"].sum()
    total_flagged = summary_df["flagged_count"].sum()
    overall_ratio = total_flagged / max(1, total_images)

    print(f"Total images scored : {total_images}")
    print(f"Total flagged       : {total_flagged}")
    print(f"Overall flagged ratio: {overall_ratio:.1%}")
    print()

    # Show per-figure results for the 3 patents with the most flagged images
    top3 = summary_df.nlargest(3, "flagged_count")[["patent_id", "total_images", "flagged_count", "flagged_ratio"]]
    print("Top 3 patents by flagged count:")
    display(top3.reset_index(drop=True))
    print()

    for patent_id in top3["patent_id"]:
        json_path = triage_path / f"{patent_id}.json"
        if not json_path.exists():
            print(f"  {patent_id}: JSON not found")
            continue
        with open(json_path) as fh:
            data = json.load(fh)
        figures_df = pd.DataFrame(data["figures"])
        print(f"\n--- {patent_id}  (total={data['total']}  flagged={data['flagged']}) ---")
        display(figures_df[["file", "pred", "score_drawing", "score_table", "keep"]])
